### Hi everyone!) I’m trying to learn using publicly available notebooks and ideas from GitHub. In this notebook, I sort the publicly available notebooks by speed, and starting with the weakest ones and moving up to the leaders, I run them and study how they play so that, in the end, I can create something of my own based on your solutions.
### I’m learning with DEEPSEEK. I have 3 dialogues in it: in one I’m studying your work, in another I’m looking at ideas from GitHub, and in the third I’m reading the discussions!))

# Kaggriculture: Game Map and Work Plan


## What this notebook is

This is a **game map, not an agent**. There is no bot code here — everything is collected in one place:

- all rules and mechanics of Kaggriculture,
- all engine constants (prices, yields, market curves),
- tables for crops, animals, shops, land, and hiring,
- season phase strategy and common beginner mistakes,
- the work plan for our agent.

**Why:** so that anyone (including myself a month from now) can open this, understand the game in 20 minutes, and see what we are doing and why.

---

## What Kaggriculture is

Two players. Each has a 10×10 farm. Only the NW quadrant 5×5 is unlocked at the start. Starting bank — **$3,000**. The season lasts **720 turns = 30 days × 24 turns**.

**Whoever ends with the most coins in the bank wins.** The shed, crops on the farm, and animals do not count. Only coins.

Each turn:
- the **farmer** performs 1 action,
- **each hired hand** performs 1 action,
- up to **10 market orders** can be issued.

---

## Contents

1. **Mechanics** — what you can actually do
2. **Crops** — all parameters + profit per tile-day
3. **Animals** — all parameters + payback time
4. **Market** — prices, curves, what crashes
5. **Town shops** — who buys what
6. **Land and hiring** — quadrants, Fibonacci, shed
7. **Life cycles** — plants and animals
8. **Season phase strategy** — days 1-5, 6-15, 16-24, 25-30
9. **Common beginner mistakes**
10. **Agent work plan** — what we do next

---

## Key facts in 30 seconds

| Item | Value |
|---|---|
| Season length | 720 turns (30 days × 24) |
| Win condition | most coins in the bank at the end |
| Best crop | Wheat (safe) / Carrot (balanced) |
| Best animal | Goose (fast) / Cow (profitable) |
| Shed | 100 items, overflow is lost |
| Hiring | Fibonacci: 1, 1, 2, 3, 5, 8, 13... per day |
| Land | NE $1k → SW $2k → SE $4k (fixed order) |
| Price floor | $1 |

---

## What this notebook does NOT cover

- Agent code (that lives in `main.py`).
- Analysis of specific replays (separate notebook).
- RL/BC training (not our path — see the plan below).

---

## Agent work plan

### Current state

- Agent: a copy of the public **V45** (first-turn wheat round trip) with our layers on top.
- LB peak: ~2700.
- Currently hovering around: ~2500 (the drop is the Elo race condition, not a weakness in the bot).
- The final result is computed with **Bradley-Terry** over all episodes, not the live Elo.

### What we already know

1. **The live LB lies** — the drop 2700 → 2500 → 2000 is partly the concurrent-episode bug.
2. **The final BT scores by win/loss pairs**, not by coin margin.
3. **Copying V45** gives a stable baseline, but no growth.
4. **RL without a GPU** is not our path — too slow.
5. **Our path:** layers on top of the copy, justified by replay analysis.

### Step by step

**Step 1. Analyze leader replays (top-10, rating 3000+)**
- Download 5-10 replays.
- Write a parser → DataFrame (money, actions, crops, animals, prices).
- Build metrics: cash curve, tile usage, action counts, market timing.
- Find 3-5 divergence points: where the leader does X and we do Y.

**Step 2. Hypotheses**
- For each divergence, state a hypothesis: "the leader does X because Y".
- Verify across several replays, not just one.

**Step 3. Layers on top of our agent**
- Each hypothesis becomes a minimal 10-50 line layer.
- Each layer can be switched on/off via `settings`.
- A failure inside a layer falls back to the base command.

**Step 4. Local testing**
- Harness: 64 seeds × 5 opponents (V39, V45, Thomas 2945, Shop Router, random).
- Metric: win rate + mean margin.
- Paired test: same seeds for A and B versions.
- Threshold: change must exceed 2 standard errors.

**Step 5. Submission**
- No more than 1 submission per day.
- Only if the local win rate improved.
- Keep the last 2 slots as a "final pair" for BT.

**Step 6. Freeze**
- Two days before the deadline (September 28) — do not touch anything.
- Watch the BT leaderboard, not the live Elo.

### What we do NOT do

- We do not write an agent from scratch.
- We do not train RL without a GPU.
- We do not chase the live LB peak.
- We do not copy a new bot every day — we stabilize what we have.

---

## Links

- [thomastschinkel: The 2945 Farm](https://www.kaggle.com/code/thomastschinkel/the-2945-farm-96-vs-the-top-10-public-bots/notebook)
- [Ahmed Berat Özer V45](https://www.kaggle.com/code/ahmedberatozer/kaggriculture-v45-first-turn-wheat-round-trip/notebook)

---

## Notes

- Mechanics fully covered — sections 1-9.
- Next: leader replays, find divergence points, write our layers.
- This notebook is updated as we go.

# Analysis of Three Winning Replays from the Champion (1st Place, ~3100 Rating)

**Analysis date:** 23.09.2026
**Source:** Kaggle dataset evgendvorkin/repley-1-mesta
**Episodes:** 112509185, 112519767, 112531901
**Champion's results:** 87,492 / 107,837 / 101,957 coins
**Margin over opponent:** ~5,000 / ~13,000 / ~5,500 coins (close games, not blowouts)

---

## 1. Money curve — S-shaped, three phases

All three replays show the same curve shape.

| Phase | Days | Money | What happens |
|---|---|---|---|
| Investment | 0–10 | 3000 → 0–400 | Everything goes into seeds, animals, workers. Cash is nearly zeroed out. |
| Explosion | 11–14 | 0 → 12,000–23,000 | First melon and carrot harvest. High price, high volume. |
| Linear growth | 14–29 | +2,000–3,000 per day | Wheat, animals, fertilizer. Steady stream. |

**Key observation:** the champion is not afraid to zero out cash. On days 0–10 he holds 0–400 coins, yet still wins. This is an aggressive bet on production that pays off on day 11.

---

## 2. What he plants — a stable portfolio

All three replays show the same set.

| Crop | Plant count | Role |
|---|---|---|
| WHEAT | 25–27 | Foundation. Animal feed, tradeable good, fuel for round-trip. |
| MELON | 5 | Main source of the day 11–14 explosion. Base price 250. |
| CARROT | 5–9 | Fast cash in early-mid game. Base price 35. |
| STRAWBERRY | 1–2 | Experimental, small share. |

**Melon — 5 tiles in all three games.** This is not random. 5 tiles times 6 units equals 30 units, close to the price crash threshold (T=300 for melon). He plays right at the edge of the crash, squeezing maximum value.

---

## 3. Market ops — the most interesting part

### 3.1. Wheat round-trip — constant, not just the opening

- SELL WHEAT 1 + BUY_PRODUCT WHEAT 1 — 7–10 times
- SELL WHEAT 5 + BUY_PRODUCT WHEAT 5 — 5 times
- SELL WHEAT 6 + BUY_PRODUCT WHEAT 5 + ... — 3 times

This is not a one-off V45 opening trick. It is a permanent strategy: sell, price drops, buy back cheaper, price recovers, repeat. Arbitrage on his own sales.

### 3.2. Fertilizer spam — the main source of cash

- SELL FERTILIZER 1 — 9 times
- SELL FERTILIZER 3 — 6 times
- SELL FERTILIZER 4 — 4 times
- SELL FERTILIZER 10 + HIRE x9
- SELL FERTILIZER 11 + HIRE x9

Fertilizer is his currency. He collects it from animals and sells it to fund worker hires. This is a non-standard move that is missing from our layers.

### 3.3. Mass hiring — 9 workers in a single turn

Example: SELL FERTILIZER 10, then HIRE, HIRE, HIRE, HIRE, HIRE, HIRE, HIRE, HIRE, HIRE.

**9 workers** hired in one turn. Cost: fib(1..9) = 1+1+2+3+5+8+13+21+34 = 88 coins. Pays off instantly. Our layers (hand_align, weed_repair) are designed for 5–6 workers and may break at 9.

### 3.4. Selling carrots in large lots

- SELL CARROT 24 + HIRE x9
- SELL CARROT 20 + HIRE x9

Carrot is the primary cash crop in early-mid game. He sells in lots of 20–24, not 5.

---

## 4. Comparison with our V56

| Aspect | Champion | Our V56 (assumed) |
|---|---|---|
| Max workers | 9 per turn | 5–6 |
| Fertilizer | Sold constantly | Likely hoarded |
| Wheat round-trip | Every few turns | Opening only |
| Carrot lot size | 20–24 | Smaller |
| Melon tiles | 5 | Fewer? |
| Cash on days 0–10 | 0–400 | More cautious |
| Strategy | Aggressive | Cautious |

---

## 5. Hypotheses for fixes

1. **Fertilizer-to-cash layer.** If fertilizer piles up and is not sold, add a sale. Potential: +500–2000 coins by season end.
2. **Aggressive hire layer.** If we hire 5–6 while the champion hires 9, increase. Hiring is cheap and pays off instantly.
3. **Wheat round-trip layer.** Constant SELL and BUY of wheat moves the price and creates arbitrage. We only do it in the opening.
4. **Big carrot lots layer.** Sell carrots in lots of 20–24, not 5.
5. **Aggressive cash management.** Do not be afraid to zero out cash on days 0–10. It pays off on day 11.

---

## 6. What to do next

1. Run the same parser on our replays (V56) to find exact divergence points.
2. Write 3–5 layers based on the hypotheses.
3. Run the harness: V56 vs V56 with layers, 64 seeds.
4. Submit only what improved the local win rate.

**Potential:** if our divergence points match these patterns, +200–400 rating is achievable within a week.

In [ ]:
import json
with open('/kaggle/input/datasets/evgendvorkin/repley-1-mesta/112509185.json') as f:
    data = json.load(f)
print(type(data))
print(list(data.keys()) if isinstance(data, dict) else len(data))
print(json.dumps(data, indent=2)[:3000])

In [ ]:
import json
with open('/kaggle/input/datasets/evgendvorkin/repley-1-mesta/112509185.json') as f:
    data = json.load(f)

# Структура steps
print("len steps:", len(data['steps']))
print("len steps[0]:", len(data['steps'][0]))
print("keys steps[0][0]:", list(data['steps'][0][0].keys()))

# Один шаг целиком (например, 100)
print(json.dumps(data['steps'][100], indent=2)[:4000])

In [ ]:
import json
import pandas as pd

def parse_replay(path):
    with open(path) as f:
        data = json.load(f)
    
    rows = []
    for step_idx, step in enumerate(data['steps']):
        for player in (0, 1):
            obs = step[player]['observation']
            action = step[player].get('action', {})
            rows.append({
                'step': step_idx,
                'day': obs['day'],
                'hour': obs['hour'],
                'player': player,
                'money': obs['farms'][player]['money'],
                'rival_money': obs['farms'][1-player]['money'],
                'farmer': action.get('farmer'),
                'hands': action.get('hands'),
                'market': action.get('market'),
                'prices': obs['market']['prices'],
                'shops': obs['town']['unlocked_shops'],
            })
    return pd.DataFrame(rows)

df = parse_replay('/kaggle/input/datasets/evgendvorkin/repley-1-mesta/112509185.json')
print(df.shape)
print(df.head())

In [ ]:
import json
import pandas as pd
from collections import Counter, defaultdict

def parse_replay(path):
    with open(path) as f:
        data = json.load(f)
    
    rows = []
    for step_idx, step in enumerate(data['steps']):
        for player in (0, 1):
            obs = step[player]['observation']
            action = step[player].get('action') or {}
            farm = obs['farms'][player]
            
            # что посажено на этом ходу
            farmer_op = action.get('farmer', ['PASS'])
            hands_ops = action.get('hands', [])
            
            rows.append({
                'step': step_idx,
                'day': obs['day'],
                'hour': obs['hour'],
                'player': player,
                'money': farm['money'],
                'rival_money': obs['farms'][1-player]['money'],
                'farmer_op': farmer_op[0] if farmer_op else 'PASS',
                'farmer_full': str(farmer_op),
                'n_hands': len(hands_ops),
                'market_ops': str(action.get('market', [])),
                'melon_price': obs['market']['prices'].get('MELON', 0),
                'wheat_price': obs['market']['prices'].get('WHEAT', 0),
                'carrot_price': obs['market']['prices'].get('CARROT', 0),
                'sheep_price': obs['market']['prices'].get('WOOL', 0),
                'n_shops': len(obs['town']['unlocked_shops']),
                'shops': str(obs['town']['unlocked_shops']),
            })
    return pd.DataFrame(rows)

for path in ['/kaggle/input/datasets/evgendvorkin/repley-1-mesta/112509185.json',
             '/kaggle/input/datasets/evgendvorkin/repley-1-mesta/112519767.json',
             '/kaggle/input/datasets/evgendvorkin/repley-1-mesta/112531901.json']:
    df = parse_replay(path)
    winner = df[df['step'] == 719].sort_values('money', ascending=False).iloc[0]['player']
    loser = 1 - winner
    
    print(f"\n=== {path.split('/')[-1]} ===")
    print(f"Winner: player {winner}, money: {df[(df['step']==719)&(df['player']==winner)]['money'].iloc[0]:.0f}")
    print(f"Loser:  player {loser}, money: {df[(df['step']==719)&(df['player']==loser)]['money'].iloc[0]:.0f}")
    
    # кривая денег по дням
    print("\nMoney by day (winner vs loser):")
    daily = df[df['hour'] == 0].pivot_table(index='day', columns='player', values='money', aggfunc='first')
    print(daily.head(30))
    
    # что сажал winner
    winner_df = df[df['player'] == winner]
    plants = winner_df[winner_df['farmer_op'] == 'PLANT']['farmer_full'].value_counts()
    print("\nWinner PLANT actions:")
    print(plants.head(20))
    
    # market ops winner
    print("\nWinner market ops (top):")
    mkt = winner_df[winner_df['market_ops'] != '[]']['market_ops'].value_counts()
    print(mkt.head(20))

# Анализ трёх победных реплеев чемпиона (1 место, рейтинг ~3100)

**Дата анализа:** 23.09.2026
**Источник:** Kaggle-датасет evgendvorkin/repley-1-mesta
**Эпизоды:** 112509185, 112519767, 112531901
**Результаты чемпиона:** 87 492 / 107 837 / 101 957 монет
**Разрыв с соперником:** ~5 000 / ~13 000 / ~5 500 монет (близкие игры, не разгром)

---

## 1. Кривая денег — S-образная, три фазы

Все три реплея показывают одинаковую форму кривой.

| Фаза | Дни | Деньги | Что происходит |
|---|---|---|---|
| Инвестиции | 0–10 | 3000 → 0–400 | Всё вкладывается в семена, животных, рабочих. Кэш почти обнуляется. |
| Взрыв | 11–14 | 0 → 12 000–23 000 | Первый урожай melon и carrot. Цена высокая, объём большой. |
| Линейный рост | 14–29 | +2 000–3 000 в день | Wheat + животные + fertilizer. Стабильный поток. |

**Ключевое наблюдение:** чемпион не боится обнулить кэш. На дне 0–10 у него 0–400 монет, но он выигрывает. Это агрессивная ставка на производство, которая окупается на 11-й день.

---

## 2. Что сажает — стабильный портфель

Во всех трёх реплеях одинаковый набор.

| Культура | Кол-во посадок | Роль |
|---|---|---|
| WHEAT | 25–27 | Основа. Корм для животных, товар, топливо для round-trip. |
| MELON | 5 | Главный источник взрыва на 11–14 день. Цена 250 за штуку. |
| CARROT | 5–9 | Быстрый кэш в early-mid game. Цена 35 за штуку. |
| STRAWBERRY | 1–2 | Экспериментально, малая доля. |

**Melon — 5 тайлов во всех трёх играх.** Это не случайность. 5 тайлов умножить на 6 единиц равно 30 единиц, что близко к порогу обвала цены (T=300 для melon). Он играет на грани обвала, выжимая максимум.

---

## 3. Market ops — самое интересное

### 3.1. Wheat round-trip — постоянно, не только в opening

- SELL WHEAT 1 + BUY_PRODUCT WHEAT 1 — 7–10 раз
- SELL WHEAT 5 + BUY_PRODUCT WHEAT 5 — 5 раз
- SELL WHEAT 6 + BUY_PRODUCT WHEAT 5 + ... — 3 раза

Это не разовая фишка V45 в opening. Это постоянная стратегия: продал, цена упала, купил дешевле, цена восстановилась, повторил. Арбитраж на собственных продажах.

### 3.2. Fertilizer spam — основной источник кэша

- SELL FERTILIZER 1 — 9 раз
- SELL FERTILIZER 3 — 6 раз
- SELL FERTILIZER 4 — 4 раза
- SELL FERTILIZER 10 + HIRE x9
- SELL FERTILIZER 11 + HIRE x9

Fertilizer у него — валюта. Он собирает его с животных и продаёт, чтобы финансировать найм рабочих. Это нестандартный ход, которого нет в наших слоях.

### 3.3. Массовый найм — 9 рабочих в один ход

Пример: SELL FERTILIZER 10, затем HIRE, HIRE, HIRE, HIRE, HIRE, HIRE, HIRE, HIRE, HIRE.

**9 рабочих** наймано в один ход. Стоимость: fib(1..9) = 1+1+2+3+5+8+13+21+34 = 88 монет. Окупается мгновенно. Наши слои (hand_align, weed_repair) рассчитаны на 5–6 рабочих и могут ломаться на 9.

### 3.4. Продажа carrot большими лотами

- SELL CARROT 24 + HIRE x9
- SELL CARROT 20 + HIRE x9

Carrot — основной кэш в early-mid game. Продаёт лотами по 20–24, а не по 5.

---

## 4. Сравнение с нашим V56

| Аспект | Чемпион | Наш V56 (предположительно) |
|---|---|---|
| Рабочих максимум | 9 в ход | 5–6 |
| Fertilizer | Постоянно продаёт | Скорее всего копит |
| Wheat round-trip | Каждые несколько ходов | Только в opening |
| Carrot лоты | 20–24 | Меньше |
| Melon тайлов | 5 | Меньше? |
| Кэш на дне 0–10 | 0–400 | Осторожнее |
| Стратегия | Агрессивная | Осторожная |

---

## 5. Гипотезы для правок

1. **Слой fertilizer в кэш.** Если у нас скапливается fertilizer и не продаётся, добавить продажу. Потенциал: плюс 500–2000 монет к концу сезона.
2. **Слой aggressive hire.** Если мы нанимаем 5–6, а чемпион 9, увеличить. Найм дёшев, окупается мгновенно.
3. **Слой wheat round-trip.** Постоянный SELL и BUY wheat двигает цену и создаёт арбитраж. У нас только в opening.
4. **Слой big carrot lots.** Продавать carrot лотами по 20–24, а не по 5.
5. **Агрессивный кэш.** Не бояться обнулять кэш в дни 0–10. Это окупается на 11-й день.

---

## 6. Что делать дальше

1. Прогнать тот же парсер на наших реплеях (V56), найти точные divergence points.
2. Написать 3–5 слоёв на основе гипотез.
3. Прогнать harness: V56 против V56 со слоями, 64 сида.
4. Сабмитить только то, что улучшило win rate локально.

**Потенциал:** если наши divergence points совпадают с этими паттернами, плюс 200–400 рейтинга достижимо за неделю.